# SSB wrapper — a tour

Search Statistikkbanken for a table, look inside it, and pull the data out as a
DataFrame. The worked example is **05803 — Endringer i befolkningen i løpet av
året**, a 1735–2026 series of births, deaths and migration.

Full reference: [`src/README.md`](../src/README.md).

In [1]:
import pandas as pd

from ssb_statistikkbanken import SSB

pd.set_option("display.max_colwidth", 90)

ssb = SSB()
ssb

SSB(language='no', timeout=30, max_workers=4, cache=True)

## 1 · Find a table

`search` does a free-text search over titles, variable names and values, and
hands back a DataFrame — so you can filter and sort it like any other frame
instead of reading a printout.

In [2]:
hits = ssb.search("befolkning", limit=10)
hits[["id", "label", "firstPeriod", "lastPeriod", "updated"]]

/tmp/ipykernel_16440/248992043.py:1: UserWarning: 306 tables matched but only 10 are shown. Raise limit= or narrow the query.
  hits = ssb.search("befolkning", limit=10)


,id,label,firstPeriod,lastPeriod,updated
0,13930,"13930: FoU-personale, befolkningen, sysselsatte og studenter, etter innvandringskatego...",2007,2024,2025-12-11T07:00:00Z
1,14476,"14476: Forskere/faglig personale, befolkningen, sysselsatte og studenter, etter foreld...",2012,2024,2026-02-05T07:00:00Z
2,07519,"07519: Samisk statistikk. Folkemengde, tettsteder og areal. Norge nord for Saltfjellet...",2008,2021,2022-02-04T07:00:00Z
3,12631,12631: Andel av befolkningen (mellom 16 og 79 år) på ferie (prosent) 2013-2025,2013,2025,2026-02-26T07:00:00Z
4,05803,05803: Endringer i befolkningen i løpet av året 1735-2026,1735,2026,2026-04-15T06:00:00Z
5,13760,"13760: Arbeidsstyrken, sysselsatte, arbeidsledige og utførte ukeverk, etter kjønn og a...",2006M01,2026M06,2026-07-21T06:00:00Z
6,14483,"14483: Personer, etter arbeidsstyrkestatus, kjønn og alder. Brudd- og sesongjusterte t...",2009K1,2026K1,2026-05-07T06:00:00Z
7,12871,"12871: Befolkning, etter kommunestørrelse, alder og kjønn 2017-2026",2017,2026,2026-02-25T07:00:00Z
8,10211,10211: Alders- og kjønnsfordeling i hele befolkningen 1846-2026,1846,2026,2026-02-25T07:00:00Z
9,05813,05813: Sivilstand og kjønnsfordeling i hele befolkningen 1769-2026,1769,2026,2026-02-25T07:00:00Z


Narrow it down the way you would any DataFrame — here, the long historical series:

In [3]:
hits.loc[hits["firstPeriod"] < "1900", ["id", "label", "firstPeriod", "lastPeriod"]]

,id,label,firstPeriod,lastPeriod
4,05803,05803: Endringer i befolkningen i løpet av året 1735-2026,1735,2026
8,10211,10211: Alders- og kjønnsfordeling i hele befolkningen 1846-2026,1846,2026
9,05813,05813: Sivilstand og kjønnsfordeling i hele befolkningen 1769-2026,1769,2026


## 2 · Look inside 05803

`describe` returns a `TableInfo` that renders as a card in Jupyter. It tells you
which variables exist, how many values each has, and which ones you *must*
select (SSB rejects the query otherwise).

In [4]:
info = ssb.describe("05803")
info

variable,label,values,mandatory,examples
ContentsCode,statistikkvariabel,14,yes,Personer: Befolkning 1. januar; Levendefodte: Levendefødte i alt; Levendefodte2: Fødte utenfor ekteskap; Dodfodte: Dødfødte; … (+10)
Tid,år,292,yes,1735: 1735; 1736: 1736; 1737: 1737; 1738: 1738; … (+288)


`values` lists the codes you actually pass to `fetch`. Nothing is truncated, and
`search=` filters when a variable has hundreds of values.

In [5]:
ssb.values("05803", "ContentsCode")

,variable,code,label,index
0,ContentsCode,Personer,Befolkning 1. januar,0
1,ContentsCode,Levendefodte,Levendefødte i alt,1
2,ContentsCode,Levendefodte2,Fødte utenfor ekteskap,2
3,ContentsCode,Dodfodte,Dødfødte,3
4,ContentsCode,Dode,Døde i alt,4
5,ContentsCode,Dode2,Døde under 1 år,5
6,ContentsCode,InngEkteskap,Inngåtte ekteskap,6
7,ContentsCode,Skilsmisse,Skilsmisser,7
8,ContentsCode,Innflyttinger,Innflyttinger,8
9,ContentsCode,Utflyttinger,Utflyttinger,9


## 3 · Fetch

One call. Pick statistics by code, give the years as a plain `range`, and the
result comes back ready to use.

In [6]:
df = ssb.fetch(
    "05803",
    contents=["Personer", "Levendefodte", "Dode", "Innflyttinger", "Utflyttinger"],
    time=range(2020, 2026),
)
df

,ContentsCode,ContentsCode_code,Tid,Tid_code,year,value
0,Befolkning 1. januar,Personer,2020,2020,2020,5367580
1,Befolkning 1. januar,Personer,2021,2021,2021,5391369
2,Befolkning 1. januar,Personer,2022,2022,2022,5425270
3,Befolkning 1. januar,Personer,2023,2023,2023,5488984
4,Befolkning 1. januar,Personer,2024,2024,2024,5550203
5,Befolkning 1. januar,Personer,2025,2025,2025,5594340
6,Levendefødte i alt,Levendefodte,2020,2020,2020,52979
7,Levendefødte i alt,Levendefodte,2021,2021,2021,56060
8,Levendefødte i alt,Levendefodte,2022,2022,2022,51480
9,Levendefødte i alt,Levendefodte,2023,2023,2023,51980


Note what you did *not* have to do. Every dimension arrives as two columns — the
readable label (`ContentsCode`) and its code (`ContentsCode_code`) — so filtering
and joining key on codes while plots keep the labels. `value` is a nullable
`Int64` and `year` an `Int16`, both derived from the table's own metadata.

In [7]:
df.dtypes

ContentsCode           str
ContentsCode_code      str
Tid                    str
Tid_code               str
year                 Int16
value                Int64
dtype: object

## 4 · Reshape

The rows are tidy long, so `pivot_table` gives the classic year × statistic view
with no cleanup in between.

In [8]:
wide = df.pivot_table(index="year", columns="ContentsCode", values="value")
wide

ContentsCode,Befolkning 1. januar,Døde i alt,Innflyttinger,Levendefødte i alt,Utflyttinger
year,,,,,
2020,5367580.0,40611.0,38071.0,52979.0,26744.0
2021,5391369.0,42002.0,53947.0,56060.0,34297.0
2022,5425270.0,45774.0,90475.0,51480.0,32536.0
2023,5488984.0,43803.0,86589.0,51980.0,34011.0
2024,5550203.0,44242.0,66077.0,54013.0,31968.0
2025,5594340.0,45040.0,55243.0,55401.0,32919.0


In [9]:
# Natural increase and net migration, straight off the wide frame
wide.assign(
    fodselsoverskudd=wide["Levendefødte i alt"] - wide["Døde i alt"],
    nettoinnvandring=wide["Innflyttinger"] - wide["Utflyttinger"],
)[["fodselsoverskudd", "nettoinnvandring"]]

ContentsCode,fodselsoverskudd,nettoinnvandring
year,,
2020,12368.0,11327.0
2021,14058.0,19650.0
2022,5706.0,57939.0
2023,8177.0,52578.0
2024,9771.0,34109.0
2025,10361.0,22324.0


## 5 · When you get a code wrong

Filters are checked against the table's metadata *before* the request is sent,
so a typo comes back as a suggestion rather than an HTTP 400.

In [10]:
from ssb_statistikkbanken import SsbError

try:
    ssb.fetch("05803", contents="Innflytting")  # missing the trailing 'er'
except SsbError as exc:
    print(f"{type(exc).__name__}: {exc}")

UnknownValueError: 'Innflytting' is not a value code for ContentsCode (statistikkvariabel) in table 05803.
  Did you mean: 'Innflyttinger' (Innflyttinger), 'Utflyttinger' (Utflyttinger)?
  Tip: ssb.values("05803", "ContentsCode")


## Cheat sheet

| Call | What you get |
|---|---|
| `ssb.search("befolkning", limit=10)` | matching tables as a DataFrame |
| `ssb.describe("05803")` | table metadata card |
| `ssb.variables("05803")` | one row per variable |
| `ssb.values("05803", "ContentsCode")` | every value code |
| `ssb.fetch("05803", contents=[...], time=range(2020, 2026))` | tidy DataFrame |
| `ssb.dataset(...)` | same, but `.to_polars()` / `.to_records()` / `.query` |
| `ssb.table("05803")` | a handle bound to one table |

Filters take a single code, a list, a `range`, a wildcard (`"03*"`) or a PxWeb
expression (`"top(5)"`). Leave the time selection out and you get the most recent
period; mandatory variables you omit are selected in full.